In [ ]:
# ============================================================
# LEARNING CURVE ANALYSIS — RESUMABLE VERSION (5-point curve)
# Adapted to: SimCLR-pretrained MobileNetV2 + CBAM classifier
# Safe against Kaggle kernel resets — saves progress after
# every single (frac, rep) run and resumes automatically.
# ============================================================
#
# Assumes the following already exist in your session, from your
# original notebook: IMG_SIZE, BATCH_SIZE, AUTOTUNE, load_image,
# cbam_block, ChannelMean, ChannelMax, class_names, num_classes,
# train_paths, train_labels, val_paths, val_labels, feature_extractor, SEED

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# ---------------- CONFIG ----------------
TRAIN_FRACTIONS = [0.2, 0.4, 0.6, 0.8, 1.0]   # 5 points
N_REPEATS = 3
CURVE_EPOCHS = 15
PATIENCE = 5

NUM_CLASSES = len(class_names)

CHECKPOINT_DIR = "/kaggle/working/learning_curve_ckpt"
RESULTS_PATH = os.path.join(CHECKPOINT_DIR, "results.json")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def build_classifier():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = feature_extractor(x, training=False)
    x = cbam_block(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def make_dataset(paths, labels, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(min(len(paths), 2000), seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds


# ---------------- RESUME LOGIC ----------------
def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            raw = json.load(f)
        return {float(k): v for k, v in raw.items()}
    return {frac: {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
            for frac in TRAIN_FRACTIONS}


def save_results(results):
    tmp_path = RESULTS_PATH + ".tmp"
    serializable = {str(k): v for k, v in results.items()}
    with open(tmp_path, "w") as f:
        json.dump(serializable, f, indent=2)
    os.replace(tmp_path, RESULTS_PATH)


results = load_results()

for frac in TRAIN_FRACTIONS:
    if frac not in results:
        results[frac] = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}

# ---------------- MAIN LOOP ----------------
val_ds_fixed = make_dataset(val_paths, val_labels, shuffle=False)

total_runs = len(TRAIN_FRACTIONS) * N_REPEATS
done_runs = sum(len(results[f]["val_acc"]) for f in TRAIN_FRACTIONS)
print(f"Resuming: {done_runs}/{total_runs} runs already completed.")

for frac in TRAIN_FRACTIONS:
    already_done = len(results[frac]["val_acc"])
    if already_done >= N_REPEATS:
        print(f"[frac={frac:.1f}] already complete ({already_done}/{N_REPEATS}) — skipping")
        continue

    for rep in range(already_done, N_REPEATS):
        if frac < 1.0:
            sub_paths, _, sub_labels, _ = train_test_split(
                train_paths, train_labels,
                train_size=frac,
                random_state=SEED + rep,
                stratify=train_labels,
            )
        else:
            sub_paths, sub_labels = train_paths, train_labels

        train_ds_sub = make_dataset(sub_paths, sub_labels, shuffle=True)

        model = build_classifier()
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE, restore_best_weights=True
        )

        history = model.fit(
            train_ds_sub,
            validation_data=val_ds_fixed,
            epochs=CURVE_EPOCHS,
            callbacks=[early_stop],
            verbose=0,
        )

        best_idx = int(np.argmin(history.history["val_loss"]))
        results[frac]["train_acc"].append(float(history.history["accuracy"][best_idx]))
        results[frac]["val_acc"].append(float(history.history["val_accuracy"][best_idx]))
        results[frac]["train_loss"].append(float(history.history["loss"][best_idx]))
        results[frac]["val_loss"].append(float(history.history["val_loss"][best_idx]))

        save_results(results)

        print(f"[frac={frac:.1f}, rep={rep+1}/{N_REPEATS}] "
              f"train_acc={results[frac]['train_acc'][-1]:.4f}  "
              f"val_acc={results[frac]['val_acc'][-1]:.4f}  "
              f"(saved to {RESULTS_PATH})")

        del model
        tf.keras.backend.clear_session()

print("All runs complete." if all(len(results[f]["val_acc"]) >= N_REPEATS for f in TRAIN_FRACTIONS)
      else "Some runs still incomplete — rerun this cell to continue.")

# ---------------- AGGREGATE ----------------
fracs = np.array(TRAIN_FRACTIONS)
train_acc_mean = np.array([np.mean(results[f]["train_acc"]) for f in fracs])
train_acc_std  = np.array([np.std(results[f]["train_acc"])  for f in fracs])
val_acc_mean   = np.array([np.mean(results[f]["val_acc"])   for f in fracs])
val_acc_std    = np.array([np.std(results[f]["val_acc"])    for f in fracs])

# ---------------- PLOT ----------------
plt.figure(figsize=(8, 6))
plt.plot(fracs, train_acc_mean, 'o-', color='blue', label='Training accuracy')
plt.fill_between(fracs, train_acc_mean - train_acc_std, train_acc_mean + train_acc_std,
                  alpha=0.15, color='blue')
plt.plot(fracs, val_acc_mean, 'o-', color='red', label='Validation accuracy')
plt.fill_between(fracs, val_acc_mean - val_acc_std, val_acc_mean + val_acc_std,
                  alpha=0.15, color='red')
plt.xlabel("Fraction of Training Data Used")
plt.ylabel("Accuracy")
plt.title("Learning Curve — SimCLR + CBAM Classifier")
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curve.png", dpi=600)
plt.show()

# ---------------- INTERPRETATION METRICS ----------------
final_gap = train_acc_mean[-1] - val_acc_mean[-1]
max_gap = np.max(train_acc_mean - val_acc_mean)
val_improvement_last_half = val_acc_mean[-1] - val_acc_mean[len(val_acc_mean) // 2]

print("=" * 55)
print("LEARNING CURVE — SUMMARY")
print("=" * 55)
print(f"Final Train Accuracy       : {train_acc_mean[-1]:.4f} ± {train_acc_std[-1]:.4f}")
print(f"Final Val Accuracy         : {val_acc_mean[-1]:.4f} ± {val_acc_std[-1]:.4f}")
print(f"Final Gap (Train - Val)    : {final_gap:.4f}")
print(f"Max Gap across all fractions: {max_gap:.4f}")
print(f"Val accuracy gain, 2nd half of curve: {val_improvement_last_half:.4f}")
print("=" * 55)

if final_gap > 0.10:
    print(" Gap > 0.10 → likely overfitting")
elif val_improvement_last_half < 0.01 and final_gap < 0.05:
    print(" Val curve has plateaued and gap is small → good generalization, "
          "more data unlikely to help much")
else:
    print("  Borderline — cross-check with the statistical/bootstrap tests")

In [ ]:
# ============================================================
# BOOTSTRAP CONFIDENCE INTERVAL — Model Generalization Estimate
# ============================================================
# Uses your final trained classifier: AgriSSL_CBAMNet_labdata.keras
# Requires: val_paths, val_labels (or test_paths, test_labels),
#           make_dataset() / load_image() from your notebook,
#           class_names, ChannelMean, ChannelMax (custom CBAM layers)

import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import json

# ---------------- CONFIG ----------------
N_BOOTSTRAP = 2000
CI_LEVEL = 0.95
SEED = 42
rng = np.random.default_rng(SEED)

# ---------------- STEP 0: LOAD TRAINED MODEL ----------------
model = tf.keras.models.load_model(
    "/kaggle/working/AgriSSL_CBAMNet_labdata.keras",
    custom_objects={
        "ChannelMean": ChannelMean,
        "ChannelMax": ChannelMax,
    }
)
model.summary()

# ---------------- STEP 1: GET PREDICTIONS ONCE ----------------
eval_ds = make_dataset(val_paths, val_labels, shuffle=False)  # or test_paths/test_labels

y_true = np.array(val_labels)
y_pred_probs = model.predict(eval_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

assert len(y_true) == len(y_pred), "Mismatch between labels and predictions length!"

n_samples = len(y_true)
print(f"Total evaluation samples: {n_samples}")

# ---------------- STEP 2: BOOTSTRAP RESAMPLING ----------------
def compute_metrics(y_t, y_p):
    return {
        "accuracy":  accuracy_score(y_t, y_p),
        "precision": precision_score(y_t, y_p, average="macro", zero_division=0),
        "recall":    recall_score(y_t, y_p, average="macro", zero_division=0),
        "f1":        f1_score(y_t, y_p, average="macro", zero_division=0),
    }

boot_metrics = {"accuracy": [], "precision": [], "recall": [], "f1": []}

for i in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_samples, n_samples)
    y_t_boot = y_true[idx]
    y_p_boot = y_pred[idx]

    m = compute_metrics(y_t_boot, y_p_boot)
    for k in boot_metrics:
        boot_metrics[k].append(m[k])

    if (i + 1) % 500 == 0:
        print(f"  bootstrap iter {i+1}/{N_BOOTSTRAP} done")

# ---------------- STEP 3: COMPUTE CI ----------------
alpha = 1 - CI_LEVEL
lower_pct = 100 * (alpha / 2)
upper_pct = 100 * (1 - alpha / 2)

point_estimate = compute_metrics(y_true, y_pred)

print("=" * 60)
print(f"BOOTSTRAP {int(CI_LEVEL*100)}% CONFIDENCE INTERVALS  (n_bootstrap={N_BOOTSTRAP})")
print("=" * 60)

results_summary = {}
for metric_name, values in boot_metrics.items():
    values = np.array(values)
    ci_low = np.percentile(values, lower_pct)
    ci_high = np.percentile(values, upper_pct)
    point = point_estimate[metric_name]

    results_summary[metric_name] = {
        "point_estimate": float(point),
        "ci_lower": float(ci_low),
        "ci_upper": float(ci_high),
        "std": float(values.std()),
    }

    print(f"{metric_name.capitalize():10s}: {point*100:.2f}%  "
          f"95% CI [{ci_low*100:.2f}%, {ci_high*100:.2f}%]  "
          f"(std={values.std()*100:.2f}%)")

print("=" * 60)

# ---------------- STEP 4: SAVE RESULTS ----------------
with open("/kaggle/working/bootstrap_ci_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("Saved to /kaggle/working/bootstrap_ci_results.json")